# LRTP Protocol Evaluation — Figure Generation
**CS3102 Practical 2** · pc1-036-l ↔ pc2-002-l · 22 Apr 2026

This notebook parses all test log files and produces publication-ready figures for the evaluation chapter.
All figures are saved to `figures/` as high-resolution PNGs and as PDFs for direct inclusion in LaTeX.

In [ ]:
import re
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter

os.makedirs('figures', exist_ok=True)

# ── Consistent style ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'legend.fontsize':  10,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'grid.linestyle':   '--',
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

# Colour palette
C_BLUE   = '#2563EB'
C_ORANGE = '#EA580C'
C_GREEN  = '#16A34A'
C_PURPLE = '#7C3AED'
C_GREY   = '#6B7280'
C_RED    = '#DC2626'

def save(name):
    plt.savefig(f'figures/{name}.png')
    plt.savefig(f'figures/{name}.pdf')
    plt.show()
    print(f'  → saved figures/{name}.{{png,pdf}}')

print('Setup complete.')

---
## 1 — Hard-coded test data
All values are extracted directly from the provided log files.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TEST SUITE SUMMARY  (from test_summary.txt)
# ══════════════════════════════════════════════════════════════════════
tests = [
    {'name': 'test-0',         'label': 'Test 0\n(basic)',           'duration_s':  4,  'passed': True},
    {'name': 'test-1',         'label': 'Test 1\n(bulk RX)',         'duration_s': 62,  'passed': True},
    {'name': 'test-2',         'label': 'Test 2\n(bulk TX)',         'duration_s': 62,  'passed': True},
    {'name': 'test-3',         'label': 'Test 3\n(bidir)',           'duration_s': 65,  'passed': True},
    {'name': 'adaptive-rto',   'label': 'Adaptive\nRTO',             'duration_s':  3,  'passed': True},
    {'name': 'large-transfer', 'label': 'Large\nTransfer',           'duration_s':  4,  'passed': True},
    {'name': 'multiple-sends', 'label': 'Multiple\nSends',           'duration_s':  3,  'passed': True},
    {'name': 'stress',         'label': 'Stress',                    'duration_s':  4,  'passed': True},
]

# ══════════════════════════════════════════════════════════════════════
# PER-TEST PCB SUMMARY  (client-side unless otherwise noted)
# ══════════════════════════════════════════════════════════════════════
pcb = {
    'test-0': {
        'duration': 0.002916, 'rtt_us': 696,  'rto_us': 1_000_000,
        'data_req_tx': 1,  'data_req_re_tx': 0,
        'data_req_rx': 1,  'data_req_bytes_tx': 1280, 'data_req_bytes_rx': 1280,
        'tx_rate_bps': 3_511_659,  'rx_rate_bps': 3_511_659,
        'close_req_re_tx': 3,
    },
    'test-1': {
        'duration': 58.496152, 'rtt_us': 699,  'rto_us': 1_000_000,
        'data_req_tx': 0,  'data_req_re_tx': 0,
        'data_req_rx': 100_000, 'data_req_bytes_rx': 128_000_000,
        'data_req_bytes_tx': 0,
        'tx_rate_bps': 0, 'rx_rate_bps': 17_505_424,
        'close_req_re_tx': 3,
    },
    'test-2': {
        'duration': 58.552700, 'rtt_us': 533,  'rto_us': 1_000_000,
        'data_req_tx': 100_000, 'data_req_re_tx': 0,
        'data_req_rx': 0, 'data_req_bytes_tx': 128_000_000,
        'data_req_bytes_rx': 0,
        'tx_rate_bps': 17_488_518, 'rx_rate_bps': 0,
        'close_req_re_tx': 0,
    },
    'test-3': {
        'duration': 61.723820, 'rtt_us': 657,  'rto_us': 1_000_000,
        'data_req_tx': 100_000, 'data_req_re_tx': 0,
        'data_req_rx': 100_000, 'data_req_bytes_tx': 128_000_000,
        'data_req_bytes_rx': 128_000_000,
        'tx_rate_bps': 16_590_029, 'rx_rate_bps': 16_590_029,
        'close_req_re_tx': 0,
    },
    'adaptive-rto': {
        'duration': 0.007879, 'rtt_us': 523,  'rto_us': 1_000_000,
        'srtt_us': 678, 'rttvar_us': 124,
        'data_req_tx': 10, 'data_req_re_tx': 0,
        'data_req_bytes_tx': 12_800,
        'tx_rate_bps': 12_996_573, 'rx_rate_bps': 0,
        'close_req_re_tx': 0,
    },
    'large-transfer': {
        'duration': 0.011176, 'rtt_us': 484,  'rto_us': 1_000_000,
        'srtt_us': 484, 'rttvar_us': 37,
        'data_req_tx': 20, 'data_req_re_tx': 0,
        'data_req_bytes_tx': 20_480,
        'tx_rate_bps': 14_659_985, 'rx_rate_bps': 0,
        'close_req_re_tx': 0,
    },
    'multiple-sends': {
        'duration': 0.003973, 'rtt_us': 402,  'rto_us': 1_000_000,
        'srtt_us': 627, 'rttvar_us': 195,
        'data_req_tx': 5, 'data_req_re_tx': 0,
        'data_req_bytes_tx': 136,
        'tx_rate_bps': 273_848, 'rx_rate_bps': 0,
        'close_req_re_tx': 0,
    },
    'stress': {
        'duration': 0.053880, 'rtt_us': 466,  'rto_us': 1_000_000,
        'srtt_us': 494, 'rttvar_us': 18,
        'data_req_tx': 100, 'data_req_re_tx': 0,
        'data_req_bytes_tx': 25_600,
        'tx_rate_bps': 3_801_039, 'rx_rate_bps': 0,
        'min_rtt_us': 404, 'max_rtt_us': 905, 'avg_rtt_us': 524,
        'close_req_re_tx': 0,
    },
}

# ══════════════════════════════════════════════════════════════════════
# PER-PACKET RTT DATA  (from log tables)
# ══════════════════════════════════════════════════════════════════════

# adaptive-rto client (10 packets, 1280 bytes each)
adaptive_rto_client = {
    'pkt':    [1,   2,   3,   4,   5,   6,   7,   8,   9,  10],
    'rtt':    [727, 790, 708, 658, 668, 661, 638, 669, 610, 523],
    'srtt':   [781, 782, 772, 757, 745, 734, 722, 715, 701, 678],
    'rttvar': [311, 235, 194, 174, 152, 135, 125, 107, 106, 124],
    'rto':    [1_000_000]*10,
}

# adaptive-rto server (10 packets)
adaptive_rto_server = {
    'pkt':    [1,   2,   3,   4,   5,   6,   7,   8,   9,  10],
    'rtt':    [725]*10,
    'srtt':   [725]*10,
    'rttvar': [362]*10,
    'rto':    [1_000_000]*10,
}

# large-transfer client (20 packets, 1024 bytes each)
large_transfer_client = {
    'pkt':    list(range(1, 21)),
    'rtt':    [639,655,653,524,489,499,485,478,448,476,476,476,477,476,445,474,447,432,467,484],
    'srtt':   [651,651,651,635,616,601,586,572,556,546,537,529,522,516,507,502,495,487,484,484],
    'rttvar': [248,187,140,136,138,132,128,123,123,112,101, 91, 81, 72, 71, 61, 59, 60, 50, 37],
    'rto':    [1_000_000]*20,
}

# multiple-sends client (5 payloads)
multiple_sends_client = {
    'pkt':    [1,   2,   3,   4,   5],
    'rtt':    [664, 600, 645, 520, 402],
    'srtt':   [698, 685, 680, 660, 627],
    'rttvar': [273, 229, 181, 175, 195],
    'rto':    [1_000_000]*5,
    'label':  ['Small(5)', 'Medium(31)', 'Large(92)', 'Binary(8)', 'Empty(0)'],
    'bytes':  [5, 31, 92, 8, 0],
}

print('Data loaded.')

---
## Figure 1 — Test suite overview: pass/fail and duration

In [ ]:
labels   = [t['label'] for t in tests]
durations = [t['duration_s'] for t in tests]
colours  = [C_GREEN if t['passed'] else C_RED for t in tests]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, durations, color=colours, edgecolor='white', linewidth=0.8, zorder=3)

for bar, d in zip(bars, durations):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{d}s', ha='center', va='bottom', fontsize=9, color='#374151')

ax.set_ylabel('Duration (s)')
ax.set_title('Figure 1 — Test Suite Results: All 8/8 Tests Passed\n'
             'pc1-036-l ↔ pc2-002-l, 22 Apr 2026')
ax.set_ylim(0, max(durations)*1.18)

pass_patch = mpatches.Patch(color=C_GREEN, label='PASSED')
ax.legend(handles=[pass_patch], loc='upper right')

# Annotate the long tests
for i, t in enumerate(tests):
    if t['duration_s'] > 10:
        ax.annotate('100 000 packets', xy=(i, t['duration_s']/2),
                    ha='center', va='center', fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
save('fig1_test_overview')

---
## Figure 2 — RTT convergence: Large-Transfer (20 packets)

In [ ]:
d = large_transfer_client

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(d['pkt'], d['rtt'],    color=C_BLUE,   marker='o', ms=5, lw=1.4,
        label='RTT (measured)', zorder=4)
ax.plot(d['pkt'], d['srtt'],   color=C_ORANGE, marker='s', ms=4, lw=2,
        label='SRTT (smoothed)', zorder=4)
ax.fill_between(d['pkt'],
                [s - v for s,v in zip(d['srtt'], d['rttvar'])],
                [s + v for s,v in zip(d['srtt'], d['rttvar'])],
                color=C_ORANGE, alpha=0.12, label='SRTT ± RTTVAR')

ax.set_xlabel('Packet number')
ax.set_ylabel('Time (μs)')
ax.set_title('Figure 2 — RTT and SRTT Convergence (Large-Transfer, 20 × 1 024 B)')
ax.legend()
ax.set_xlim(0.5, 20.5)

# Annotate RTTVAR collapse
ax.annotate('RTTVAR: 248 μs → 37 μs\n(path stabilises)',
            xy=(20, d['srtt'][-1]), xytext=(14, 620),
            arrowprops=dict(arrowstyle='->', color=C_GREY),
            fontsize=9, color=C_GREY)

plt.tight_layout()
save('fig2_rtt_convergence_large_transfer')

---
## Figure 3 — Adaptive RTO: RTT, SRTT, RTTVAR (Adaptive-RTO test)

In [ ]:
d = adaptive_rto_client

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

# Top: RTT and SRTT
ax1.plot(d['pkt'], d['rtt'],  color=C_BLUE,   marker='o', ms=5, lw=1.4, label='RTT')
ax1.plot(d['pkt'], d['srtt'], color=C_ORANGE, marker='s', ms=4, lw=2,   label='SRTT')
ax1.fill_between(d['pkt'],
                 [s - v for s,v in zip(d['srtt'], d['rttvar'])],
                 [s + v for s,v in zip(d['srtt'], d['rttvar'])],
                 color=C_ORANGE, alpha=0.12, label='SRTT ± RTTVAR')
ax1.set_ylabel('Time (μs)')
ax1.set_title('Figure 3 — Adaptive RTO Test: RTT Estimates and RTTVAR Decay\n'
              '(10 × 1 280 B packets, client side)')
ax1.legend(loc='upper right')

# Bottom: RTTVAR
ax2.bar(d['pkt'], d['rttvar'], color=C_PURPLE, alpha=0.75, label='RTTVAR')
ax2.set_xlabel('Packet number')
ax2.set_ylabel('RTTVAR (μs)')
ax2.legend()
ax2.set_xlim(0.5, 10.5)

plt.tight_layout()
save('fig3_adaptive_rto_estimates')

---
## Figure 4 — Fixed RTO vs Adaptive RTO: computed RTO trajectory

In [ ]:
# The measured RTO stays clamped at 1 000 000 μs throughout all tests
# because SRTT + 4*RTTVAR never exceeds 1 s on the lab LAN.
# We compute what the adaptive RTO *would* be if the floor were removed,
# to illustrate the convergence behaviour.

d = large_transfer_client  # 20 packets — best illustration of convergence

computed_rto = [s + 4*v for s, v in zip(d['srtt'], d['rttvar'])]
fixed_rto    = [1_000_000] * len(d['pkt'])          # RFC 6298 minimum
clamped_rto  = [max(1_000_000, r) for r in computed_rto]  # actual applied

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.axhline(1_000_000, color=C_RED, lw=1.5, ls='--', label='Fixed RTO (1 000 000 μs = 1 s)', zorder=2)
ax.plot(d['pkt'], computed_rto, color=C_BLUE, marker='o', ms=5, lw=1.8,
        label='Adaptive RTO = SRTT + 4·RTTVAR (unclamped)', zorder=4)
ax.plot(d['pkt'], clamped_rto,  color=C_GREEN, marker='^', ms=5, lw=1.8, ls=':',
        label='Adaptive RTO (clamped ≥ 1 s, as applied)', zorder=4)

ax.set_xlabel('Packet number')
ax.set_ylabel('RTO (μs)')
ax.set_title('Figure 4 — Fixed vs Adaptive RTO: Computed Timeout Trajectory\n'
             '(Large-Transfer, 20 × 1 024 B; lab LAN latency ~0.5 ms)')
ax.legend()
ax.set_ylim(0, 1_200_000)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x/1000)} ms'))

ax.annotate('Adaptive RTO converges to\n~1484 μs (well below 1 s floor)',
            xy=(20, computed_rto[-1]), xytext=(13, 300_000),
            arrowprops=dict(arrowstyle='->', color=C_GREY),
            fontsize=9, color=C_GREY)

plt.tight_layout()
save('fig4_fixed_vs_adaptive_rto')

---
## Figure 5 — Throughput comparison across all tests

In [ ]:
# Gather TX and RX rates (Mbps) per test
test_names = ['test-0', 'test-1', 'test-2', 'test-3',
              'adaptive-rto', 'large-transfer', 'multiple-sends', 'stress']
short_labels = ['Test 0', 'Test 1\n(bulk RX)', 'Test 2\n(bulk TX)', 'Test 3\n(bidir)',
                'Adaptive\nRTO', 'Large\nTransfer', 'Multiple\nSends', 'Stress']

tx_mbps = [pcb[t].get('tx_rate_bps', 0) / 1e6 for t in test_names]
rx_mbps = [pcb[t].get('rx_rate_bps', 0) / 1e6 for t in test_names]

x = np.arange(len(test_names))
w = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, tx_mbps, w, label='TX throughput', color=C_BLUE,   alpha=0.85, zorder=3)
b2 = ax.bar(x + w/2, rx_mbps, w, label='RX throughput', color=C_ORANGE, alpha=0.85, zorder=3)

def annotate_bars(bars):
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.2,
                    f'{h:.1f}', ha='center', va='bottom', fontsize=8)

annotate_bars(b1)
annotate_bars(b2)

ax.set_xticks(x)
ax.set_xticklabels(short_labels)
ax.set_ylabel('Throughput (Mbps)')
ax.set_title('Figure 5 — Mean TX and RX Throughput per Test (client-side PCB)')
ax.legend()
ax.set_ylim(0, 22)

plt.tight_layout()
save('fig5_throughput_comparison')

---
## Figure 6 — Final SRTT and RTTVAR across multi-packet tests

In [ ]:
mp_tests  = ['adaptive-rto', 'large-transfer', 'multiple-sends', 'stress']
mp_labels = ['Adaptive RTO\n(10 pkts)', 'Large Transfer\n(20 pkts)',
             'Multiple Sends\n(5 pkts)', 'Stress\n(100 pkts)']
final_srtt   = [pcb[t]['srtt_us']   for t in mp_tests]
final_rttvar = [pcb[t]['rttvar_us'] for t in mp_tests]

x = np.arange(len(mp_tests))
w = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
b1 = ax.bar(x - w/2, final_srtt,   w, label='Final SRTT',   color=C_ORANGE, alpha=0.85, zorder=3)
b2 = ax.bar(x + w/2, final_rttvar, w, label='Final RTTVAR', color=C_PURPLE, alpha=0.85, zorder=3)

for bar, val in zip(list(b1)+list(b2), final_srtt+final_rttvar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(mp_labels)
ax.set_ylabel('Time (μs)')
ax.set_title('Figure 6 — Converged SRTT and RTTVAR at End of Each Multi-Packet Test')
ax.legend()

plt.tight_layout()
save('fig6_final_srtt_rttvar')

---
## Figure 7 — RTT evolution across all three multi-packet tests (overlaid)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

# Large transfer — normalise to fraction of run
lt = large_transfer_client
ax.plot([p/20 for p in lt['pkt']], lt['rtt'],
        color=C_BLUE, marker='.', ms=6, lw=1.4, label='Large-Transfer (20 × 1 024 B)')
ax.plot([p/20 for p in lt['pkt']], lt['srtt'],
        color=C_BLUE, lw=1.8, ls='--', alpha=0.6)

# Adaptive RTO
ar = adaptive_rto_client
ax.plot([p/10 for p in ar['pkt']], ar['rtt'],
        color=C_ORANGE, marker='.', ms=6, lw=1.4, label='Adaptive-RTO (10 × 1 280 B)')
ax.plot([p/10 for p in ar['pkt']], ar['srtt'],
        color=C_ORANGE, lw=1.8, ls='--', alpha=0.6)

# Multiple sends
ms = multiple_sends_client
ax.plot([p/5 for p in ms['pkt']], ms['rtt'],
        color=C_GREEN, marker='.', ms=6, lw=1.4, label='Multiple-Sends (5 payloads)')
ax.plot([p/5 for p in ms['pkt']], ms['srtt'],
        color=C_GREEN, lw=1.8, ls='--', alpha=0.6)

ax.set_xlabel('Normalised progress through test (0 = first packet, 1 = last)')
ax.set_ylabel('RTT / SRTT (μs)')
ax.set_title('Figure 7 — RTT (solid) and SRTT (dashed) Across Three Tests\n'
             'Normalised to fraction of run to allow direct comparison')
ax.legend()
ax.set_xlim(0, 1.05)

plt.tight_layout()
save('fig7_rtt_all_tests_normalised')

---
## Figure 8 — Multiple-Sends: per-payload RTT breakdown

In [ ]:
ms = multiple_sends_client

fig, ax = plt.subplots(figsize=(8, 4))

bar_colours = [C_BLUE, C_ORANGE, C_GREEN, C_PURPLE, C_GREY]
bars = ax.bar(ms['label'], ms['rtt'], color=bar_colours, alpha=0.85, zorder=3, edgecolor='white')

for bar, r, b in zip(bars, ms['rtt'], ms['bytes']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{r} μs', ha='center', va='bottom', fontsize=9)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            f'{b} B', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.set_ylabel('RTT (μs)')
ax.set_title('Figure 8 — Multiple-Sends: Per-Payload RTT\n'
             '(payload size labelled in white; RTT overhead independent of size)')
ax.set_ylim(0, 780)

plt.tight_layout()
save('fig8_multiple_sends_rtt')

---
## Figure 9 — Stress test: RTT statistics summary

In [ ]:
s = pcb['stress']

fig, ax = plt.subplots(figsize=(7, 4))

categories = ['Min RTT', 'Avg RTT', 'Final RTT', 'SRTT', 'Max RTT']
values     = [s['min_rtt_us'], s['avg_rtt_us'], s['rtt_us'], s['srtt_us'], s['max_rtt_us']]
colours    = [C_GREEN, C_BLUE, C_ORANGE, C_PURPLE, C_RED]

bars = ax.barh(categories, values, color=colours, alpha=0.82, edgecolor='white', zorder=3)
for bar, v in zip(bars, values):
    ax.text(v + 8, bar.get_y() + bar.get_height()/2,
            f'{v} μs', va='center', fontsize=10)

ax.set_xlabel('Time (μs)')
ax.set_title('Figure 9 — Stress Test: RTT Statistics\n'
             '(100 packets × 256 B, no loss, no retransmissions)')
ax.set_xlim(0, 1100)
ax.invert_yaxis()

plt.tight_layout()
save('fig9_stress_rtt_stats')

---
## Figure 10 — Retransmission analysis across all tests

In [ ]:
test_names_all = ['test-0', 'test-1', 'test-2', 'test-3',
                  'adaptive-rto', 'large-transfer', 'multiple-sends', 'stress']
short_labels_all = ['Test 0', 'Test 1', 'Test 2', 'Test 3',
                    'Adaptive\nRTO', 'Large\nTransfer', 'Multiple\nSends', 'Stress']

data_re_tx   = [pcb[t].get('data_req_re_tx', 0) for t in test_names_all]
close_re_tx  = [pcb[t].get('close_req_re_tx', 0) for t in test_names_all]

x = np.arange(len(test_names_all))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 4.5))
b1 = ax.bar(x - w/2, data_re_tx,  w, label='data_req retransmissions',  color=C_BLUE,   alpha=0.85, zorder=3)
b2 = ax.bar(x + w/2, close_re_tx, w, label='close_req retransmissions', color=C_ORANGE, alpha=0.85, zorder=3)

for bar, v in zip(list(b1)+list(b2), data_re_tx+close_re_tx):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.05,
                str(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(short_labels_all)
ax.set_ylabel('Retransmission count')
ax.set_title('Figure 10 — Retransmissions per Test\n'
             '(data_req: 0 across all tests; close_req: 3 in tests 0 & 1 — teardown race)')
ax.legend()
ax.set_ylim(0, 5)

# Annotate the close_req retransmits
ax.annotate('Teardown race:\nclose_req retransmitted\n3× before ACK received',
            xy=(0.5, 3), xytext=(3.5, 4),
            arrowprops=dict(arrowstyle='->', color=C_GREY),
            fontsize=9, color=C_GREY, ha='center')

plt.tight_layout()
save('fig10_retransmissions')

---
## Figure 11 — Fixed RTO floor: gap between adaptive estimate and applied RTO

In [ ]:
# Show all three multi-packet test computed RTOs on one plot
# to quantify how much headroom the 1s floor provides on a LAN

lt  = large_transfer_client
ar  = adaptive_rto_client

lt_computed  = [s + 4*v for s,v in zip(lt['srtt'], lt['rttvar'])]
ar_computed  = [s + 4*v for s,v in zip(ar['srtt'], ar['rttvar'])]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(lt['pkt'], lt_computed, color=C_BLUE, marker='o', ms=5, lw=1.6,
        label='Large-Transfer: SRTT + 4·RTTVAR')
ax.plot(ar['pkt'], ar_computed, color=C_ORANGE, marker='s', ms=5, lw=1.6,
        label='Adaptive-RTO: SRTT + 4·RTTVAR')

ax.axhline(1_000_000, color=C_RED, lw=2, ls='--', label='RFC 6298 floor (1 s)', zorder=2)

# Shade the gap
ax.fill_between(lt['pkt'], lt_computed, [1_000_000]*len(lt['pkt']),
                color=C_BLUE, alpha=0.08, label='Headroom (floor − computed)')

ax.set_xlabel('Packet number')
ax.set_ylabel('RTO (μs)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x/1000)} ms'))
ax.set_title('Figure 11 — Computed Adaptive RTO vs RFC 6298 Minimum Floor\n'
             'On a LAN with sub-ms RTT the adaptive estimate is far below the 1 s minimum')
ax.legend()
ax.set_ylim(0, 1_200_000)

plt.tight_layout()
save('fig11_rto_floor_gap')

---
## Figure 12 — Dashboard: key metrics summary (single-page overview)

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.38)

# ── (a) Test durations ────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, :])
durs = [t['duration_s'] for t in tests]
lbls = [t['label'] for t in tests]
bars_a = ax_a.bar(lbls, durs, color=[C_GREEN]*8, alpha=0.82, edgecolor='white', zorder=3)
for bar, d in zip(bars_a, durs):
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
              f'{d}s', ha='center', fontsize=8)
ax_a.set_ylabel('Duration (s)')
ax_a.set_title('(a) Test durations — all PASSED')
ax_a.set_ylim(0, 80)

# ── (b) Throughput ────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1, :2])
x_b  = np.arange(len(test_names))
ax_b.bar(x_b - 0.2, tx_mbps, 0.38, label='TX', color=C_BLUE,   alpha=0.82, zorder=3)
ax_b.bar(x_b + 0.2, rx_mbps, 0.38, label='RX', color=C_ORANGE, alpha=0.82, zorder=3)
ax_b.set_xticks(x_b)
ax_b.set_xticklabels(short_labels, fontsize=8)
ax_b.set_ylabel('Mbps')
ax_b.set_title('(b) Throughput (Mbps)')
ax_b.legend(fontsize=8)

# ── (c) Final SRTT per multi-packet test ─────────────────────────────
ax_c = fig.add_subplot(gs[1, 2])
ax_c.barh(mp_labels, final_srtt, color=C_ORANGE, alpha=0.82, zorder=3)
for i, v in enumerate(final_srtt):
    ax_c.text(v + 5, i, f'{v} μs', va='center', fontsize=8)
ax_c.set_xlabel('SRTT (μs)')
ax_c.set_title('(c) Final SRTT')
ax_c.invert_yaxis()

# ── (d) RTT convergence (large transfer) ─────────────────────────────
ax_d = fig.add_subplot(gs[2, :2])
ax_d.plot(lt['pkt'], lt['rtt'],  color=C_BLUE,   lw=1.4, label='RTT')
ax_d.plot(lt['pkt'], lt['srtt'], color=C_ORANGE, lw=2,   label='SRTT')
ax_d.fill_between(lt['pkt'],
                  [s-v for s,v in zip(lt['srtt'],lt['rttvar'])],
                  [s+v for s,v in zip(lt['srtt'],lt['rttvar'])],
                  color=C_ORANGE, alpha=0.12)
ax_d.set_xlabel('Packet')
ax_d.set_ylabel('μs')
ax_d.set_title('(d) RTT convergence — Large-Transfer')
ax_d.legend(fontsize=8)

# ── (e) Retransmissions ───────────────────────────────────────────────
ax_e = fig.add_subplot(gs[2, 2])
ax_e.bar(short_labels_all, close_re_tx, color=C_ORANGE, alpha=0.82, zorder=3)
ax_e.set_xticklabels(short_labels_all, fontsize=7, rotation=30, ha='right')
ax_e.set_ylabel('Count')
ax_e.set_title('(e) close_req re-tx')

fig.suptitle('LRTP Evaluation Dashboard — pc1-036-l ↔ pc2-002-l, 22 Apr 2026',
             fontsize=13, fontweight='bold', y=0.99)

save('fig12_dashboard')

---
## Summary — figure file listing

In [ ]:
import glob
print('Generated files:')
for f in sorted(glob.glob('figures/*')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f:<50} {size_kb:6.1f} KB')

print('\nLaTeX inclusion example:')
print(r'''
\begin{figure}[H]
    \centering
    \includegraphics[width=0.90\textwidth]{figures/fig4_fixed_vs_adaptive_rto}
    \caption{Fixed vs adaptive RTO: computed timeout trajectory ...}
    \label{fig:fixed_vs_adaptive_rto}
\end{figure}
''')